In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/wine-grade-predicting-wine-quality-with-machine-learning/sample_submission.csv
/kaggle/input/competitions/wine-grade-predicting-wine-quality-with-machine-learning/train.csv
/kaggle/input/competitions/wine-grade-predicting-wine-quality-with-machine-learning/test.csv


In [2]:
from scipy.optimize import minimize
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import cohen_kappa_score
import lightgbm as lgb
import warnings
warnings.filterwarnings('ignore')

In [3]:
train = pd.read_csv('/kaggle/input/competitions/wine-grade-predicting-wine-quality-with-machine-learning/train.csv')
test = pd.read_csv('/kaggle/input/competitions/wine-grade-predicting-wine-quality-with-machine-learning/test.csv')

In [4]:
def engineer_wine_features(df):
    df = df.copy()
    
    # Asitlik İlişkileri
    if 'fixed acidity' in df.columns and 'volatile acidity' in df.columns:
        df['total_acidity'] = df['fixed acidity'] + df['volatile acidity']
        df['acidity_ratio'] = df['fixed acidity'] / (df['volatile acidity'] + 1e-5)
    
    # Sülför Dioksit İlişkileri
    if 'free sulfur dioxide' in df.columns and 'total sulfur dioxide' in df.columns:
        df['free_so2_ratio'] = df['free sulfur dioxide'] / (df['total sulfur dioxide'] + 1e-5)
        df['bound_so2'] = df['total sulfur dioxide'] - df['free sulfur dioxide']
        
    # Şeker ve Alkol Oranları
    if 'residual sugar' in df.columns and 'alcohol' in df.columns:
        df['sugar_alcohol_ratio'] = df['residual sugar'] / (df['alcohol'] + 1e-5)
        
    return df

In [5]:
train_df = engineer_wine_features(train)
test_df = engineer_features(test) if 'engineer_features' in locals() else engineer_wine_features(test)

X = train_df.drop(columns=['Id', 'quality'], errors='ignore')
y = train_df['quality']
X_test = test_df.drop(columns=['Id'], errors='ignore')

In [6]:
class OptimizedRounder:
    def __init__(self):
        self.coef_ = 0

    def _kappa_loss(self, coef, X, y):
        X_p = np.copy(X)
        for i, pred in enumerate(X_p):
            if pred < coef[0]:
                X_p[i] = 3
            elif pred < coef[1]:
                X_p[i] = 4
            elif pred < coef[2]:
                X_p[i] = 5
            elif pred < coef[3]:
                X_p[i] = 6
            elif pred < coef[4]:
                X_p[i] = 7
            else:
                X_p[i] = 8
        return -cohen_kappa_score(y, X_p, weights='quadratic')

    def fit(self, X, y):
        initial_coef = [3.5, 4.5, 5.5, 6.5, 7.5]
        res = minimize(self._kappa_loss, initial_coef, args=(X, y), method='Nelder-Mead')
        self.coef_ = res.x

    def predict(self, X):
        X_p = np.copy(X)
        res = np.zeros(len(X_p), dtype=int)
        for i, pred in enumerate(X_p):
            if pred < self.coef_[0]:
                res[i] = 3
            elif pred < self.coef_[1]:
                res[i] = 4
            elif pred < self.coef_[2]:
                res[i] = 5
            elif pred < self.coef_[3]:
                res[i] = 6
            elif pred < self.coef_[4]:
                res[i] = 7
            else:
                res[i] = 8
        return res

In [7]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof_preds = np.zeros(len(X))
test_preds = np.zeros(len(X_test))

lgb_params = {
    'objective': 'regression', # Ordinal QWK için regresyon kullanılır
    'metric': 'rmse',
    'learning_rate': 0.03,
    'num_leaves': 31,
    'random_state': 42,
    'verbose': -1
}